# 04 — Drift Analysis

Computes **all drift metrics** from Chapter 3 of the thesis for each window pair (A, B) across all supported model types, then produces a combined temporal dashboard.

**Model types:** `MODEL_TYPES = ['xgboost', 'logreg']`  
Artifacts are read from `data/models/{model_type}/` and `data/shap/{model_type}/`.

**Input:** `data/processed/`, `data/windows/`, `data/models/{model_type}/`, `data/shap/{model_type}/`  
**Output per model type:** `data/results/{model_type}/drift_metrics.csv`  
**Combined outputs:** `data/results/combined/drift_metrics_combined.csv`, `data/results/combined/temporal_drift_report.png`

---

| Metric | Symbol | Section | XGBoost | LR |
|--------|--------|---------|---------|-----|
| Covariate drift | Δ_X | §3.2 | ✓ | ✓ |
| Target drift | Δ_Y | §3.2 | ✓ | ✓ |
| Performance change | Δ_perf | §3.2 | ✓ | ✓ |
| Local dynamic drift (cosine) | Δ_E^{loc}(cos) | §3.5 | SHAP-based | coeff-vector-based |
| Local dynamic drift (RBO) | Δ_E^{loc}(rbo) | §3.5 | SHAP-based | coeff-vector-based |
| Within-window baseline (cosine) | σ_E^{pool}(cos) | §3.6 | SHAP-based | coeff-vector-based |
| Within-window baseline (RBO) | σ_E^{pool}(rbo) | §3.6 | SHAP-based | coeff-vector-based |
| Drift Ratio (cosine) | DR_cos | §3.7 | ✓ | ✓ |
| Drift Ratio (RBO) | DR_rbo | §3.7 | ✓ | ✓ |
| Global drift | Δ_E^{glob} | §3.8 | SHAP ranking | \|coef\| ranking |
| Explainer stochasticity | — | §3.9 | ✓ | NaN |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


> **Setup note:** this notebook requires the `rbo` package for rank-biased overlap metrics (§3.5, §3.8).  
> If not already installed, run: `pip install rbo`

In [4]:
%pip install rbo

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import rbo

WORKSPACE  = Path('/content/drive/MyDrive/Thesis/Shoppers_workspace')
PROC_DIR   = WORKSPACE / 'data' / 'processed'
WIN_DIR    = WORKSPACE / 'data' / 'windows'

# ── Per-model-type base directories ───────────────────────────────────
MODEL_DIR_BASE   = WORKSPACE / 'data' / 'models'
SHAP_DIR_BASE    = WORKSPACE / 'data' / 'shap'
LIME_DIR_BASE    = WORKSPACE / 'data' / 'lime'
RESULTS_DIR_BASE = WORKSPACE / 'data' / 'results'

# Aliases used by downstream cells
SHAP_DIR    = SHAP_DIR_BASE / 'xgboost'
RESULTS_DIR = RESULTS_DIR_BASE

COMBINED_DIR = RESULTS_DIR_BASE / 'combined'
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

# ── What to compute ────────────────────────────────────────────────────
MODEL_TYPES = ['xgboost', 'logreg', 'mlp_plr']
EXPLAINERS  = ['shap', 'lime']   # per-row attribution sources to iterate over
                                 # LR uses its coefficients as a built-in 'coef' explainer
                                 # (emitted always regardless of EXPLAINERS content)

RBO_P = 0.9
EPS   = 1e-8

print('Imports OK')
print(f'MODEL_TYPES: {MODEL_TYPES}')
print(f'EXPLAINERS : {EXPLAINERS}')


In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names     = feature_names_json['all']   # flat list, 119 names in column order
num_feature_names = feature_names_json['num']   # 113 numeric feature names

# Column indices of numeric / binary features in X (column order matches feature_names)
num_col_idx = [feature_names.index(fn) for fn in num_feature_names]
bin_col_idx = [i for i in range(len(feature_names)) if i not in set(num_col_idx)]

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']
n_features = len(feature_names)

print(f'X: {X.shape}, features: {n_features}, R={R}, pairs: {len(pairs)}')
print(f'  Numeric: {len(num_col_idx)}, Binary: {len(bin_col_idx)}')

X: (160057, 119), features: 119, R=8, pairs: 2
  Numeric: 113, Binary: 6


## Distance and RBO helper functions

In [ ]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    """d_cos(u,v) = 1 - cosine_similarity(u,v).
    Both-zero → 0.0; one-zero one-nonzero → np.nan (excluded by nanmedian aggregation).
    """
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u < 1e-12 and norm_v < 1e-12:
        return 0.0
    if norm_u < 1e-12 or norm_v < 1e-12:
        # One attribution vector is exactly zero; the other is not.
        # Not a meaningful distance — marked nan and dropped during aggregation.
        return np.nan
    return float(1.0 - np.dot(u, v) / (norm_u * norm_v))


def rbo_score_local(l1, l2, p=0.9):
    """Simple RBO implementation as fallback."""
    if not l1 or not l2: return 0.0
    n = min(len(l1), len(l2))
    s1, s2 = set(), set()
    agreement = 0.0
    res = 0.0
    for i in range(n):
        s1.add(l1[i])
        s2.add(l2[i])
        agreement = len(s1.intersection(s2)) / (i + 1)
        res += agreement * (p ** i)
    return res * (1 - p)


def rbo_distance(u: np.ndarray, v: np.ndarray, p: float = RBO_P) -> float:
    """
    d_rbo(u, v) = 1 - RBO(rank(u), rank(v)).
    Features are ranked by decreasing absolute attribution.
    Both-zero → 0.0; one-zero one-nonzero → np.nan.
    """
    u_zero = np.all(np.abs(u) < 1e-12)
    v_zero = np.all(np.abs(v) < 1e-12)
    if u_zero and v_zero:
        return 0.0
    if u_zero or v_zero:
        return np.nan
    rank_u = list(np.argsort(-np.abs(u)))
    rank_v = list(np.argsort(-np.abs(v)))
    
    if rbo is not None:
        score = rbo.RankingSimilarity(rank_u, rank_v).rbo(p=p)
    else:
        score = rbo_score_local(rank_u, rank_v, p=p)
        
    return float(1.0 - score)


def instance_dynamic_drift(
    phi_A: np.ndarray,  # (R, p) — R attribution vectors from window A
    phi_B: np.ndarray,  # (R, p) — R attribution vectors from window B
    dist_fn,
) -> float:
    """
    δ_dyn(x; d) = mean over all R×R cross-window pairs of d(φ_A^{(r)}, φ_B^{(s)}).
    """
    dists = []
    for r in range(R):
        for s in range(R):
            dists.append(dist_fn(phi_A[r], phi_B[s]))
    return float(np.nanmean(dists))


def instance_baseline_instability(
    phi: np.ndarray,  # (R, p) — R attribution vectors from same window
    dist_fn,
) -> float:
    """
    δ^{base}(x; d) = median over all r < r' pairs of d(φ^{(r)}, φ^{(r')}).
    """
    dists = []
    for r, r2 in combinations(range(R), 2):
        dists.append(dist_fn(phi[r], phi[r2]))
    return float(np.nanmedian(dists)) if dists else 0.0


def rbo_global_drift(
    phi_bar_A: np.ndarray,  # (|F|, p)
    phi_bar_B: np.ndarray,  # (|F|, p)
    p: float = RBO_P,
) -> float:
    """
    Δ_E^{glob}: 1 − RBO(rank(g_A), rank(g_B)) where g(j) = mean_x |φ̄_j(x)|.
    """
    g_A = np.abs(phi_bar_A).mean(axis=0)
    g_B = np.abs(phi_bar_B).mean(axis=0)
    rank_A = list(np.argsort(-g_A))
    rank_B = list(np.argsort(-g_B))
    
    if rbo is not None:
        score = rbo.RankingSimilarity(rank_A, rank_B).rbo(p=p)
    else:
        score = rbo_score_local(rank_A, rank_B, p=p)
        
    return float(1.0 - score)


print('Distance functions defined.')

Distance functions defined.


## Main drift computation loop

For each window pair we compute all metrics and append to a results list.

## Logistic Regression — coefficient-drift metric mapping

LR coefficients are the explanation (no post-hoc XAI). The table below documents which SHAP-based metrics have coefficient analogues and which do not.

| SHAP metric | LR analogue | Notes |
|---|---|---|
| Covariate drift Δ_X | ✓ identical | data-level, model-agnostic |
| Target drift Δ_Y | ✓ identical | data-level, model-agnostic |
| Performance change Δ_perf | ✓ identical | same PR-AUC formula |
| `loc_cos` / `loc_rbo` | mean `cosine_distance(coef_A[r], coef_B[s])` over R×R pairs | no per-instance dimension for LR |
| `base_cos_A/B`, `sigma_cos` | median `cosine_distance(coef[r], coef[r′])` over C(R,2) pairs per window | same formula, coefficient vectors as input |
| Drift Ratio | ✓ identical formula | |
| `global_rbo` | `1 − RBO(rank(|coef̄_A|), rank(|coef̄_B|))` | ranked by mean absolute coefficient |
| `shap_stochasticity` | NaN | LR coefficients are deterministic |

In [ ]:
all_results = {}


def _load_attributions(explainer_name, pair_dir, attr_dir, pred_data):
    """Return (attr_A, attr_B, flagged_subset_idx_within_flagged) or None on missing files.

    For SHAP: attr files are (R, |F|, p) and the returned subset-index is the identity
    (range(|F|)), because SHAP runs on the full flagged set.

    For LIME: attr files are (R, |F_lime|, p) and the subset-index is the saved
    lime_flagged_idx.npy (positions within the full flagged set).
    """
    if explainer_name == 'shap':
        a_path = attr_dir / 'shap_A.npy'
        b_path = attr_dir / 'shap_B.npy'
        if not (a_path.exists() and b_path.exists()):
            return None
        a = np.load(a_path)
        b = np.load(b_path)
        subset = np.arange(a.shape[1], dtype=np.int64)
        return a, b, subset

    elif explainer_name == 'lime':
        a_path       = attr_dir / 'lime_A.npy'
        b_path       = attr_dir / 'lime_B.npy'
        subset_path  = attr_dir / 'lime_flagged_idx.npy'
        if not (a_path.exists() and b_path.exists() and subset_path.exists()):
            return None
        a = np.load(a_path)
        b = np.load(b_path)
        subset = np.load(subset_path)
        return a, b, subset

    raise ValueError(f'Unknown explainer: {explainer_name}')


def _attribution_drift_metrics(attr_A: np.ndarray, attr_B: np.ndarray) -> dict:
    """Compute §3.5 / §3.6 / §3.7 / §3.8 metrics from a pair of (R, n_sub, p) arrays."""
    n_sub = attr_A.shape[1]
    out   = {}

    # §3.5 Dynamic cross-window drift
    dyn_cos_per_instance = []
    dyn_rbo_per_instance = []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        dyn_cos_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, cosine_distance))
        dyn_rbo_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, rbo_distance))
    out['loc_cos'] = float(np.nanmedian(dyn_cos_per_instance))
    out['loc_rbo'] = float(np.nanmedian(dyn_rbo_per_instance))

    # §3.6 Within-window baseline instability
    base_cos_A_per, base_cos_B_per = [], []
    base_rbo_A_per, base_rbo_B_per = [], []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        base_cos_A_per.append(instance_baseline_instability(phi_A_i, cosine_distance))
        base_cos_B_per.append(instance_baseline_instability(phi_B_i, cosine_distance))
        base_rbo_A_per.append(instance_baseline_instability(phi_A_i, rbo_distance))
        base_rbo_B_per.append(instance_baseline_instability(phi_B_i, rbo_distance))
    sigma_A_cos = float(np.nanmedian(base_cos_A_per))
    sigma_B_cos = float(np.nanmedian(base_cos_B_per))
    sigma_A_rbo = float(np.nanmedian(base_rbo_A_per))
    sigma_B_rbo = float(np.nanmedian(base_rbo_B_per))
    out['base_cos_A'] = sigma_A_cos
    out['base_cos_B'] = sigma_B_cos
    out['base_rbo_A'] = sigma_A_rbo
    out['base_rbo_B'] = sigma_B_rbo
    out['sigma_cos']  = 0.5 * (sigma_A_cos + sigma_B_cos)
    out['sigma_rbo']  = 0.5 * (sigma_A_rbo + sigma_B_rbo)

    # §3.7 Drift Ratio
    out['drift_ratio_cos'] = out['loc_cos'] / (out['sigma_cos'] + EPS)
    out['drift_ratio_rbo'] = out['loc_rbo'] / (out['sigma_rbo'] + EPS)

    # §3.8 Global attribution drift
    phi_bar_A = attr_A.mean(axis=0)
    phi_bar_B = attr_B.mean(axis=0)
    out['global_rbo'] = rbo_global_drift(phi_bar_A, phi_bar_B)

    return out


def _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx):
    """Per-pair metrics that are explainer-independent (§3.2 signals)."""
    row = {
        'model_type':    mt,
        'explainer':     None,   # filled in later
        'pair_id':       pid,
        'step_label_A':  p['step_label_A'],
        'step_label_B':  p['step_label_B'],
        'n_train_A':     p['n_train_A'],
        'n_train_B':     p['n_train_B'],
        'n_eval':        p['n_eval'],
        'n_flagged':     len(flagged_local_idx),
        'pr_auc_A':      float(pred_data['pr_auc_A']),
        'pr_auc_B':      float(pred_data['pr_auc_B']),
    }

    # §3.2 Performance change
    row['delta_perf'] = (1.0 - float(pred_data['pr_auc_A'])) - (1.0 - float(pred_data['pr_auc_B']))

    # §3.2 Target drift
    row['delta_Y'] = abs(float(Y[idx_A].mean()) - float(Y[idx_B].mean()))

    # §3.2 Covariate drift
    pair_dir = MODEL_DIR_BASE / mt / f'pair_{pid:02d}'
    reference_scaler_path = pair_dir / 'reference_scaler.joblib'
    if reference_scaler_path.exists():
        reference_scaler = joblib.load(reference_scaler_path)
        X_A_num_sc = reference_scaler.transform(X[idx_A][:, num_col_idx])
        X_B_num_sc = reference_scaler.transform(X[idx_B][:, num_col_idx])
        w1_per_feat = np.zeros(n_features, dtype=np.float64)
        for k, j in enumerate(num_col_idx):
            w1_per_feat[j] = wasserstein_distance(X_A_num_sc[:, k], X_B_num_sc[:, k])
        for j in bin_col_idx:
            w1_per_feat[j] = wasserstein_distance(X[idx_A][:, j], X[idx_B][:, j])
        row['delta_X'] = float(w1_per_feat.mean())
    else:
        row['delta_X'] = np.nan

    return row


# ═════════════════════════════════════════════════════════════════════════════
# Main loop
# ═════════════════════════════════════════════════════════════════════════════

for mt in MODEL_TYPES:
    print(f'\n{"="*60}')
    print(f'  Model type: {mt}')
    print(f'{"="*60}')

    mt_model_dir = MODEL_DIR_BASE / mt
    results = []

    for p in pairs:
        pid      = p['pair_id']
        pair_dir = mt_model_dir / f'pair_{pid:02d}'

        print(f'\n── [{mt}] Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]} ──')

        pred_data_path = pair_dir / 'predictions.npz'
        if not pred_data_path.exists():
            print(f'  WARNING: predictions.npz not found for pair {pid:02d}. Skipping.')
            continue

        pred_data = np.load(pred_data_path)
        idx_A    = np.array(p['idx_A'],    dtype=np.int64)
        idx_B    = np.array(p['idx_B'],    dtype=np.int64)
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)
        flagged_local_idx = pred_data['flagged_idx']
        n_flagged = len(flagged_local_idx)
        print(f'  Flagged: {n_flagged:,}')

        base_row = _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx)
        print(f'  Perf: Δ={base_row["delta_perf"]:+.4f}   Target drift: {base_row["delta_Y"]:.4f}   Cov drift: {base_row["delta_X"]:.4f}')

        # ─────────────────────────────────────────────────────────────
        # LR coefficient-drift path — always emitted; explainer='coef'
        # ─────────────────────────────────────────────────────────────
        if mt == 'logreg':
            coef_A_path = pair_dir / 'coef_A.npy'
            coef_B_path = pair_dir / 'coef_B.npy'
            if coef_A_path.exists() and coef_B_path.exists():
                coef_A = np.load(coef_A_path)
                coef_B = np.load(coef_B_path)

                row = dict(base_row)
                row['explainer'] = 'coef'
                row['shap_stochasticity'] = np.nan   # not applicable

                # §3.5 analogue
                dists_cos = [cosine_distance(coef_A[r], coef_B[s])
                             for r in range(R) for s in range(R)]
                dists_rbo = [rbo_distance(coef_A[r], coef_B[s])
                             for r in range(R) for s in range(R)]
                row['loc_cos'] = float(np.nanmean(dists_cos))
                row['loc_rbo'] = float(np.nanmean(dists_rbo))

                # §3.6 analogue
                sigma_A_cos = float(np.nanmedian([cosine_distance(coef_A[r], coef_A[r2])
                                                  for r, r2 in combinations(range(R), 2)]))
                sigma_B_cos = float(np.nanmedian([cosine_distance(coef_B[r], coef_B[r2])
                                                  for r, r2 in combinations(range(R), 2)]))
                sigma_A_rbo = float(np.nanmedian([rbo_distance(coef_A[r], coef_A[r2])
                                                  for r, r2 in combinations(range(R), 2)]))
                sigma_B_rbo = float(np.nanmedian([rbo_distance(coef_B[r], coef_B[r2])
                                                  for r, r2 in combinations(range(R), 2)]))
                row['base_cos_A'] = sigma_A_cos
                row['base_cos_B'] = sigma_B_cos
                row['base_rbo_A'] = sigma_A_rbo
                row['base_rbo_B'] = sigma_B_rbo
                row['sigma_cos']  = 0.5 * (sigma_A_cos + sigma_B_cos)
                row['sigma_rbo']  = 0.5 * (sigma_A_rbo + sigma_B_rbo)

                # §3.7 Drift Ratio
                row['drift_ratio_cos'] = row['loc_cos'] / (row['sigma_cos'] + EPS)
                row['drift_ratio_rbo'] = row['loc_rbo'] / (row['sigma_rbo'] + EPS)

                # §3.8 Global coef drift
                g_A = np.abs(coef_A.mean(axis=0))
                g_B = np.abs(coef_B.mean(axis=0))
                rank_A = list(np.argsort(-g_A))
                rank_B = list(np.argsort(-g_B))
                row['global_rbo'] = float(1.0 - rbo.RankingSimilarity(rank_A, rank_B).rbo(p=RBO_P))

                print(f'  [coef]   loc_cos={row["loc_cos"]:.4f}  DR_cos={row["drift_ratio_cos"]:.3f}  '
                      f'loc_rbo={row["loc_rbo"]:.4f}  global_rbo={row["global_rbo"]:.4f}')
                results.append(row)
            else:
                print(f'  WARNING: coef_A.npy/coef_B.npy not found for LR pair {pid:02d}')

        # ─────────────────────────────────────────────────────────────
        # Explainer-based rows — one per explainer in EXPLAINERS
        # ─────────────────────────────────────────────────────────────
        for explainer_name in EXPLAINERS:
            # Skip SHAP for LR (not computed)
            if explainer_name == 'shap' and mt == 'logreg':
                continue

            if explainer_name == 'shap':
                attr_dir = SHAP_DIR_BASE / mt / f'pair_{pid:02d}'
            elif explainer_name == 'lime':
                attr_dir = LIME_DIR_BASE / mt / f'pair_{pid:02d}'
            else:
                print(f'  WARNING: unknown explainer {explainer_name!r}')
                continue

            loaded = _load_attributions(explainer_name, pair_dir, attr_dir, pred_data)
            if loaded is None:
                print(f'  WARNING: {explainer_name} files not found for pair {pid:02d}. Skipping.')
                continue
            attr_A, attr_B, subset = loaded

            row = dict(base_row)
            row['explainer'] = explainer_name
            row['n_subset']  = int(len(subset))

            # §3.9 Stochasticity diagnostic — same JSON schema for both explainers
            stoch_path = attr_dir / 'stochasticity.json'
            if stoch_path.exists():
                with open(stoch_path) as f:
                    stoch_data = json.load(f)
                row['shap_stochasticity'] = float(stoch_data['max_abs_diff'])
            else:
                row['shap_stochasticity'] = np.nan
                print(f'  WARNING: stochasticity.json not found under {attr_dir.name}')

            if attr_A.shape[1] == 0:
                for k in ['loc_cos', 'loc_rbo', 'base_cos_A', 'base_cos_B', 'base_rbo_A',
                          'base_rbo_B', 'sigma_cos', 'sigma_rbo',
                          'drift_ratio_cos', 'drift_ratio_rbo', 'global_rbo']:
                    row[k] = np.nan
                print(f'  [{explainer_name}] No subset instances — drift metrics NaN.')
                results.append(row)
                continue

            metrics = _attribution_drift_metrics(attr_A, attr_B)
            row.update(metrics)

            print(f'  [{explainer_name:<4s}]  loc_cos={row["loc_cos"]:.4f}  DR_cos={row["drift_ratio_cos"]:.3f}  '
                  f'loc_rbo={row["loc_rbo"]:.4f}  global_rbo={row["global_rbo"]:.4f}  '
                  f'stoch={row["shap_stochasticity"]:.2e}')

            results.append(row)

    all_results[mt] = results

print('\n✓ All drift metrics computed.')


## Save results

In [ ]:
# ── Save per-model-type CSVs (all explainers combined per model type) ─────────
for mt in MODEL_TYPES:
    if mt not in all_results or not all_results[mt]:
        continue
    mt_results_dir = RESULTS_DIR_BASE / mt
    mt_results_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_results[mt]).to_csv(mt_results_dir / 'drift_metrics.csv', index=False)
    print(f'Saved {mt}/drift_metrics.csv  ({len(all_results[mt])} rows)')

# ── Combined CSV ──────────────────────────────────────────────────────────────
combined_df = pd.concat(
    [pd.DataFrame(all_results[mt]) for mt in MODEL_TYPES
     if mt in all_results and all_results[mt]],
    ignore_index=True,
)
combined_df.to_csv(COMBINED_DIR / 'drift_metrics_combined.csv', index=False)
print(f'Saved combined/drift_metrics_combined.csv  ({len(combined_df)} rows)')

# ── Display summary per (model_type, explainer) ───────────────────────────────
if not combined_df.empty:
    print('\n── Rows per (model_type, explainer) ──')
    print(combined_df.groupby(['model_type', 'explainer']).size().to_string())

    print('\n── Per-pair summary (xgboost, shap) ──')
    sub = combined_df.query("model_type == 'xgboost' and explainer == 'shap'")
    if not sub.empty:
        cols = [c for c in [
            'pair_id', 'step_label_A', 'step_label_B',
            'pr_auc_A', 'pr_auc_B', 'delta_perf', 'delta_X', 'delta_Y',
            'loc_cos', 'sigma_cos', 'drift_ratio_cos',
            'loc_rbo', 'sigma_rbo', 'drift_ratio_rbo',
            'global_rbo', 'shap_stochasticity',
        ] if c in sub.columns]
        print(sub[cols].to_string(index=False))

# ── Backwards-compat alias ─────────────────────────────────────────────────────
drift_df = combined_df.query("model_type == 'xgboost' and explainer == 'shap'").copy() \
           if not combined_df.empty else pd.DataFrame()


## §3.10 Reporting: time-series plots

In [ ]:
# Plot time series of drift metrics. Default: compare model types for a given
# explainer ('shap' preferred; fall back to 'coef' if SHAP missing).
# To plot LIME results instead, set REPORT_EXPLAINER = 'lime' and re-run.
REPORT_EXPLAINER = 'shap'

df_all = combined_df.copy()
# For LR there's no SHAP — include its 'coef' rows when REPORT_EXPLAINER == 'shap'
# so the panel still shows LR alongside XGBoost and MLP-PLR for continuity.
if REPORT_EXPLAINER == 'shap':
    df_all = df_all[(df_all['explainer'] == 'shap') |
                    ((df_all['model_type'] == 'logreg') & (df_all['explainer'] == 'coef'))]
else:
    df_all = df_all[df_all['explainer'] == REPORT_EXPLAINER]

df_all = df_all.dropna(subset=['loc_cos'])

if df_all.empty:
    print(f'No rows for explainer={REPORT_EXPLAINER!r} — skipping time-series plot.')
else:
    pair_ids = sorted(df_all['pair_id'].unique())
    x_idx    = list(range(len(pair_ids)))

    _ref_mt = next((mt for mt in MODEL_TYPES if mt in df_all['model_type'].values), MODEL_TYPES[0])
    _ref_df = df_all[df_all['model_type'] == _ref_mt].sort_values('pair_id')
    x_labels = list(_ref_df['step_label_A'])

    COLORS  = {'xgboost': 'crimson',  'logreg': 'steelblue', 'mlp_plr': 'darkviolet'}
    MARKERS = {'xgboost': 'o',        'logreg': 's',         'mlp_plr': '^'}
    LINES   = {'xgboost': '-',        'logreg': '--',        'mlp_plr': '-.'}

    fig = plt.figure(figsize=(14, 14))
    gs  = gridspec.GridSpec(4, 1, hspace=0.45)

    # Panel 1: contextual signals
    ax1  = fig.add_subplot(gs[0])
    ax1b = ax1.twinx()
    df0 = df_all[df_all['model_type'] == _ref_mt].sort_values('pair_id')
    ax1.plot(x_idx, df0['delta_X'].values, 'o-',  color='steelblue',  label='Cov. drift (Δ_X)')
    ax1.plot(x_idx, df0['delta_Y'].values, 's--', color='darkorange', label='Target drift (Δ_Y)')
    for mt in MODEL_TYPES:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty: continue
        ax1b.plot(x_idx, dft['delta_perf'].values, f'{MARKERS[mt]}:', color=COLORS[mt],
                  label=f'Δ_perf ({mt})')
    ax1.set_ylabel('Drift magnitude')
    ax1b.set_ylabel('Perf. change')
    ax1.set_title(f'Contextual signals over time  ({REPORT_EXPLAINER})')
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1b.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8)
    ax1.set_xticks(x_idx); ax1.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # Panel 2: local drift vs baselines
    ax2 = fig.add_subplot(gs[1])
    for mt in MODEL_TYPES:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty: continue
        ax2.plot(x_idx, dft['loc_cos'].values, f'{MARKERS[mt]}{LINES[mt]}',
                 color=COLORS[mt], label=f'Loc. drift ({mt})')
        ax2.plot(x_idx, dft['sigma_cos'].values, f'{MARKERS[mt]}:',
                 color=COLORS[mt], alpha=0.5, label=f'Baseline σ ({mt})')
    ax2.set_ylabel('Cosine distance')
    ax2.set_title(f'Local explanation drift vs. within-window baseline (cosine)  ({REPORT_EXPLAINER})')
    ax2.legend(fontsize=7)
    ax2.set_xticks(x_idx); ax2.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # Panel 3: drift ratio
    ax3 = fig.add_subplot(gs[2])
    for mt in MODEL_TYPES:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty: continue
        ax3.plot(x_idx, dft['drift_ratio_cos'].values, f'{MARKERS[mt]}{LINES[mt]}',
                 color=COLORS[mt], label=f'DR_cos ({mt})')
    ax3.axhline(1.0, color='grey', linewidth=0.7, linestyle=':')
    ax3.set_ylabel('Drift ratio (cos)')
    ax3.set_title(f'Drift ratio over time  ({REPORT_EXPLAINER})')
    ax3.legend(fontsize=7)
    ax3.set_xticks(x_idx); ax3.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # Panel 4: global RBO drift
    ax4 = fig.add_subplot(gs[3])
    for mt in MODEL_TYPES:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty: continue
        ax4.plot(x_idx, dft['global_rbo'].values, f'{MARKERS[mt]}{LINES[mt]}',
                 color=COLORS[mt], label=f'Global RBO drift ({mt})')
    ax4.set_ylabel('1 − RBO')
    ax4.set_title(f'Global attribution ranking drift  ({REPORT_EXPLAINER})')
    ax4.legend(fontsize=7)
    ax4.set_xticks(x_idx); ax4.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    plt.tight_layout()
    plt.savefig(COMBINED_DIR / f'drift_time_series_{REPORT_EXPLAINER}.png', dpi=150)
    plt.show()


## Per-feature SHAP drift profile

Shows which features shift most in global importance across the timeline.

In [ ]:
# Per-feature attribution over time. Set REPORT_MT and REPORT_EXP to select which
# (model_type, explainer) combination to visualise. Default: XGBoost + SHAP.
REPORT_MT  = 'xgboost'
REPORT_EXP = 'shap'

attr_base_dir = SHAP_DIR_BASE if REPORT_EXP == 'shap' else LIME_DIR_BASE
attr_dir_mt   = attr_base_dir / REPORT_MT

file_A = 'shap_A.npy' if REPORT_EXP == 'shap' else 'lime_A.npy'

global_imp_matrix = []
pair_labels = []
for p in pairs:
    pid = p['pair_id']
    ap = attr_dir_mt / f'pair_{pid:02d}' / file_A
    if not ap.exists():
        continue
    attr_A = np.load(ap)
    phi_bar_A = attr_A.mean(axis=0)
    g_A = np.abs(phi_bar_A).mean(axis=0)
    global_imp_matrix.append(g_A)
    pair_labels.append(p['step_label_A'])

if global_imp_matrix:
    imp_arr = np.array(global_imp_matrix)
    feat_variance = imp_arr.std(axis=0)
    top_feat_idx = np.argsort(-feat_variance)[:15]

    fig, ax = plt.subplots(figsize=(12, 5))
    for j in top_feat_idx:
        ax.plot(range(len(pair_labels)), imp_arr[:, j], marker='o',
                label=feature_names[j], linewidth=1.5)
    ax.set_title(f'Global {REPORT_EXP.upper()} importance over time — top 15 variable features  ({REPORT_MT})')
    ax.set_xlabel('Window pair (A end date)')
    ax.set_ylabel(f'Mean |{REPORT_EXP.upper()}|')
    ax.set_xticks(range(len(pair_labels)))
    ax.set_xticklabels(pair_labels, rotation=30, ha='right', fontsize=7)
    ax.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout()
    out_dir = RESULTS_DIR_BASE / REPORT_MT
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / f'feature_importance_over_time_{REPORT_EXP}.png', dpi=150)
    plt.show()
else:
    print(f'No attribution files found for {REPORT_MT}/{REPORT_EXP}.')


## Summary statistics

In [ ]:
numeric_cols = [
    'pr_auc_A', 'pr_auc_B', 'delta_perf',
    'delta_X', 'delta_Y',
    'loc_cos', 'base_cos_A', 'base_cos_B', 'sigma_cos', 'drift_ratio_cos',
    'loc_rbo', 'base_rbo_A', 'base_rbo_B', 'sigma_rbo', 'drift_ratio_rbo',
    'global_rbo', 'shap_stochasticity',
]

if not combined_df.empty:
    for (mt, exp), group_df in combined_df.groupby(['model_type', 'explainer']):
        avail_cols = [c for c in numeric_cols if c in group_df.columns]
        summary = group_df[avail_cols].describe().T[['mean', 'std', 'min', 'max']]
        print(f'\n=== {mt} / {exp} — overall summary ({len(group_df)} pairs) ===')
        print(summary.round(4).to_string())

        if 'drift_ratio_cos' in group_df.columns:
            dr_cos_mean = group_df['drift_ratio_cos'].mean()
            print(f'\n  Mean Drift Ratio (cosine): {dr_cos_mean:.3f}')
            if dr_cos_mean > 2.0:
                print('  → Temporal drift substantially exceeds retraining noise.')
            elif dr_cos_mean > 1.0:
                print('  → Temporal drift moderately exceeds retraining baseline.')
            else:
                print('  → Temporal drift is within the range of retraining noise.')
